# 1 - Imports

In [3]:
%reload_ext autoreload
%autoreload 2

In [4]:
import sys
from pathlib import Path

sys.path.append(str(Path().resolve().parents[0]))

In [5]:
from src.utils import templates, config, io

/Users/hippolytegrandet/Desktop/Dev/country_risk_rating/.venv/lib/python3.9/site-packages/pypdf/_crypt_providers/_cryptography.py:32: CryptographyDeprecationWarning: ARC4 has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.ARC4 and will be removed from cryptography.hazmat.primitives.ciphers.algorithms in 48.0.0.
  from cryptography.hazmat.primitives.ciphers.algorithms import AES, ARC4


In [6]:
import pandas as pd
import numpy as np
import xgboost as xgb

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

XGBoostError: 
XGBoost Library (libxgboost.dylib) could not be loaded.
Likely causes:
  * OpenMP runtime is not installed
    - vcomp140.dll or libgomp-1.dll for Windows
    - libomp.dylib for Mac OSX
    - libgomp.so for Linux and other UNIX-like OSes
    Mac OSX users: Run `brew install libomp` to install OpenMP runtime.

  * You are running 32-bit Python on a 64-bit OS

Error message(s): ["dlopen(/Users/hippolytegrandet/Desktop/Dev/country_risk_rating/.venv/lib/python3.9/site-packages/xgboost/lib/libxgboost.dylib, 0x0006): Library not loaded: @rpath/libomp.dylib\n  Referenced from: <FBD6AEF9-AFAB-39D7-B881-755157DA0497> /Users/hippolytegrandet/Desktop/Dev/country_risk_rating/.venv/lib/python3.9/site-packages/xgboost/lib/libxgboost.dylib\n  Reason: tried: '/usr/local/opt/libomp/lib/libomp.dylib' (no such file), '/System/Volumes/Preboot/Cryptexes/OS/usr/local/opt/libomp/lib/libomp.dylib' (no such file), '/usr/local/opt/libomp/lib/libomp.dylib' (no such file), '/System/Volumes/Preboot/

# 2 - Preprocessing

In [11]:
from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.preprocessing import OneHotEncoder, LabelEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split

In [32]:
X = io.load_csv(config.PROCESSED_DATA_DIR / 'X.csv', index_col=0)
y = io.load_csv(config.PROCESSED_DATA_DIR / 'y.csv', index_col=0)

In [33]:
numeric_pipeline = Pipeline([
    ("imputer", KNNImputer(n_neighbors=5)),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    ))
])

In [34]:
categorical_cols = X.columns[X.dtypes == 'object']
numeric_cols = X.columns[X.dtypes != 'object']

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_cols),
    ("cat", categorical_pipeline, categorical_cols)
])

In [35]:
y_transformer = LabelEncoder()

In [36]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25,
)

In [37]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed  = preprocessor.transform(X_test)

In [42]:
y_train_transformed = y_transformer.fit_transform(y_train)
y_test_transformed  = y_transformer.transform(y_test)

/Users/hippolytegrandet/Desktop/Dev/country_risk_rating/.venv/lib/python3.9/site-packages/sklearn/preprocessing/_label.py:110: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/Users/hippolytegrandet/Desktop/Dev/country_risk_rating/.venv/lib/python3.9/site-packages/sklearn/preprocessing/_label.py:129: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, dtype=self.classes_.dtype, warn=True)


# 3 - Train Model

## 3.1 - Baseline, Logistic Regression Model

In [43]:
from sklearn.linear_model import LogisticRegression

In [44]:
lr_model_pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("model", LogisticRegression())
])

In [48]:
lr_model_pipeline.fit(X_train, y_train)

/Users/hippolytegrandet/Desktop/Dev/country_risk_rating/.venv/lib/python3.9/site-packages/sklearn/utils/validation.py:1408: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/Users/hippolytegrandet/Desktop/Dev/country_risk_rating/.venv/lib/python3.9/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   KNNImputer()),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  Index(['YEAR', 'GE.EST', 'NY.GDP.PCAP.CD', 'DT.DOD.DLXF.CD', 'DT.DOD.DIMF.CD',
       'SP.URB.TOTL.IN.ZS', 'SP.POP.TOTL', 'FS.AST.PRVT.GD.ZS',
       'DT.DOD.DECT.GN.ZS', 'FR.INR.LEND', 'EN.URB.LCTY.UR.ZS', 'SP.POP.DPND',
       'NY.GDP.MKT...
       'NV.SRV.TOTL.KD.ZG', 'NY.GNP.MKTP.KD.ZG', 'NY.GDP.MKTP.KD.ZG',
       'BX.KLT.DINV.WD.GD.ZS', 'NV.IND.TOTL.KD.ZG'],
      dtype='object')),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore',
                                                                                 sparse_output=False))]),
                                                  Index(['ISO3_COUNTRY_CODE', 'GEO_REGION', 'ADMIN_REGION', 'LENDING_TYPE',
       'INCOME_GROUP'],
      dtype='object'))])),
                ('model', LogisticRegression())])

In [49]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)


In [55]:
# Predictions
y_pred = lr_model_pipeline.predict(X_test)
y_proba = lr_model_pipeline.predict_proba(X_test)[:, 1]

# Evaluation metrics
results = {
    "accuracy": accuracy_score(y_test, y_pred),
    "precision": precision_score(y_test, y_pred, average='micro'),
    "recall": recall_score(y_test, y_pred, average='micro'),
    "f1": f1_score(y_test, y_pred, average='micro'),
    # "roc_auc": roc_auc_score(y_test, y_proba, multi_class='ovr')
}

print("Baseline Logistic Regression Results")
for k, v in results.items():
    print(f"{k}: {v:.4f}")

print("\nClassification Report")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix")
print(confusion_matrix(y_test, y_pred))

Baseline Logistic Regression Results
accuracy: 0.7707
precision: 0.7707
recall: 0.7707
f1: 0.7707

Classification Report
              precision    recall  f1-score   support

           1       0.89      0.92      0.90       226
           2       0.81      0.82      0.81        77
           3       0.69      0.77      0.73        94
           4       0.59      0.55      0.57        66
           5       0.67      0.35      0.46       103
           6       0.62      0.64      0.63       166
           7       0.83      0.91      0.87       319

    accuracy                           0.77      1051
   macro avg       0.73      0.71      0.71      1051
weighted avg       0.76      0.77      0.76      1051


Confusion Matrix
[[208   8   0   1   0   2   7]
 [  6  63   8   0   0   0   0]
 [  7   7  72   5   1   0   2]
 [  3   0  16  36   6   4   1]
 [  4   0   6  11  36  35  11]
 [  3   0   2   8   9 106  38]
 [  4   0   0   0   2  24 289]]


## 3.2 XGBoost Classifier

In [ ]:
clf = xgb.XGBClassifier(
    reg_alpha= 0.01,
    colsample_bytree=0.60,
    eta=0.3,
    eval_metric=['mlogloss'],
    gamma=0.00001,
    reg_lambda=1.04,
    max_depth=6,
    min_child_weight=0.2,
    num_class=7,
    objective='multi:softprob',
    subsample=0.73
)

clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

print(accuracy_score(y_test, y_pred))